# RSNA knee: ROI classification, inference (rsna-knee-weights-1)

One streaming pass per GPU (`src/infer/pipeline.py`):

0. **Input normalisation**: plane names; compressed / multi-frame / RGB / duplicated-slice / spacing-less DICOMs rewritten as plain files.
1. **Read** (DataLoader workers, the only CPU work): DICOM headers of every slice, pixels of the <= 64 slices kept.
2. **Preprocess** (GPU): percentile normalisation, 384 px cache, 32 x 128 x 128 volume, in the training left-right convention.
3. **Side** (GPU): left / right for every series (a study can hold both knees).
4. **ROIs**: series flipped to a right knee, ROI U-Net, one box per ROI (ACL, MCL, menisci, the three OA compartments, Baker's) with a confidence; weak boxes replaced by registration from the same knee's other series; axial boxes registered from the sagittal / coronal ones.
5. **ROI classification**: crops, the 7 crop classifiers, combined per finding by `recipe.json` (mean rank where several models).

The 4 findings without an ROI (effusion, synovitis, contusion, fracture) stay at 0.5.

In [ ]:
import glob
import json
import os
import sys
import time

import pandas as pd
import torch

ON_KAGGLE = os.path.exists("/kaggle/input")


def kaggle_input(name, required):
    # an attached dataset, mounted as /kaggle/input/<name> or /kaggle/input/{datasets,notebooks}/<owner>/<name>
    for d in [f"/kaggle/input/{name}", *sorted(glob.glob(f"/kaggle/input/*/*/{name}"))]:
        if glob.glob(os.path.join(d, required)):
            return d
    raise FileNotFoundError(f"attach {name} (no {required})")


if ON_KAGGLE:
    # Code and wheels come from attached datasets; the competition image lacks
    # dynamic_network_architectures (the ROI U-Net) and some DICOM decoders.
    CODE = kaggle_input("rsna-knee-code", "src")
    WHEELS = kaggle_input("rsna-knee-wheels", "dynamic_network_architectures-*.whl")
    !cp -r {CODE}/src ./
    !pip install --quiet --no-index --no-deps --find-links {WHEELS} dynamic-network-architectures
    # DICOM decoders, where the wheels dataset has them (compressed series are skipped without)
    !pip install --quiet --no-index --find-links {WHEELS} pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm || true
    SRC = os.path.abspath("./src")
    DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/"
    WEIGHTS = kaggle_input("rsna-knee-weights-1", "recipe.json")
    OUT_DIR = "/tmp/knee/"
else:
    SRC = os.path.abspath("../src")
    DATA_DIR = "../input/"
    WEIGHTS = "../output/weights_1/"           # src/reproduce.sh package
    OUT_DIR = "../output/inference/"

# The pipeline runs as `<this python> -m infer.pipeline` with src/ on the path.
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, SRC)
print("src:", SRC, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      "| CPUs:", os.cpu_count())

In [ ]:
SERIES_DIR = "test_series"                       # <SERIES_DIR>/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
SERIES_CSV = os.path.join(DATA_DIR, "test_series.csv")   # series -> study, plane (folders walked if absent)
GPUS = list(range(torch.cuda.device_count()))    # one process per GPU
WORKERS = max(1, os.cpu_count() // len(GPUS))    # DICOM reading processes per GPU
LIMIT = None                                     # first N studies only (debugging)

recipe = json.load(open(os.path.join(WEIGHTS, "recipe.json")))
for w in ("side", "roi") + tuple(f"crops/{m}" for m in {m for ms in recipe.values() for m in ms}):
    assert os.path.exists(os.path.join(WEIGHTS, w, "ckpt_final.pt")), f"missing weights: {w}"
os.makedirs(OUT_DIR, exist_ok=True)
DATA_DIR, WEIGHTS, OUT_DIR = map(os.path.abspath, (DATA_DIR, WEIGHTS, OUT_DIR))
print("recipe:", recipe)

In [ ]:
t0 = time.time()
cmd = (f"{sys.executable} -m infer.pipeline --data_dir {DATA_DIR} --series_dir {SERIES_DIR} --series_csv {SERIES_CSV}"
       f" --weights {WEIGHTS} --out_dir {OUT_DIR} --submission submission.csv"
       f" --gpus {','.join(map(str, GPUS))} --workers {WORKERS}" + (f" --limit {LIMIT}" if LIMIT else ""))
print(cmd)
!{cmd}
print(f"{time.time() - t0:.0f}s")

In [ ]:
shards = sorted(glob.glob(os.path.join(OUT_DIR, "shard*", "series.csv")))
series = pd.concat([pd.read_csv(p) for p in shards], ignore_index=True)
print(series.plane.value_counts().to_dict(), "| failed:", int((series.status != "ok").sum()),
      "| sides:", series.side.value_counts().to_dict())
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()